<div style="text-align: center;">
    <h1> </font> <font color = #4854E8>Risk Passenger Rule Generation for Individual Flights</h1> </font>
</div>



**This project focuses on developing a dynamic AI model designed to automatically generate rules for identifying high-risk passengers on individual flights. By leveraging comprehensive Staging travel dataset, including flight details, personal information, and behavioral patterns, the model aims to enhance risk assessment through adaptive, data-driven rule generation.** 

<div style="text-align: left;">
    <h3> </font> <font color = #00BFFF>01- Importing Required Libraries</h3> </font>
</div>

In [1]:
import numpy as np
import pandas as pd

import os
import pickle

from datetime import datetime

pd.options.display.max_columns = None
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans  

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules
import ast

import gym
from gym import spaces
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

import warnings
warnings.filterwarnings("ignore")

<div style="text-align: left;">
    <h3> </font> <font color = #00BFFF>02- Loading the Preproceesded Staging dataset</h3> </font>
</div>

The function returns the deserialized data, which is stored in the variable data. This allows the user to load a previously saved preprocessed staging dataset into memory for further processing or analysis.

In [2]:
# Load data from the pickle file

def load_from_pickle(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    return data

data = load_from_pickle('data.pkl')

<div style="text-align: left;">
    <h3> </font> <font color = #00BFFF>03- Designing the Pipeline</h3> </font>
</div>

This pipeline is designed to process and analyze flight data to identify potential risks and anomalies in passenger behavior.

It begins with feature engineering, where a variety of relevant features are derived from raw flight data, such as age, travel document validity, visa expiration, booking history, and travel frequency. These features are then encoded to facilitate machine learning and anomaly detection. The encoded data is passed through clustering and anomaly detection methods, utilizing techniques like K-Means clustering and Isolation Forest to identify abnormal patterns and potential risks. Additionally, an Autoencoder model is used for further anomaly detection, offering a deeper layer of analysis. Finally, association rule mining is applied to uncover relationships between various features and anomalies, helping to generate actionable insights for risk assessment.

#### Integrating the pipeline into the backend

- **We can integrate the pipeline into the backend by setting up an API, to handle requests and process the data dynamically. The API would receive flight data in JSON format, apply the various steps in this pipeline (including feature engineering, encoding, clustering, and anomaly detection), and return the results as a JSON response.**

- **With this approach, the backend API can process incoming data in real-time, making it scalable and adaptable for integration with other services or user interfaces, such as dashboards or automated alerts, for real-time decision-making.**



In [3]:
def feature_engineering(flight_data):
    
    # Drop duplicates based on 'passenger_id'
    flight_data = flight_data.drop_duplicates(subset='passenger_id')

    # Feature engineering
    flight_data['age'] = (flight_data['arrival_date'] - flight_data['dob']).dt.days // 365
    flight_data['travel_document_validity_days'] = (flight_data['travelDocument_expiryDate'] - flight_data['arrival_date']).dt.days
    flight_data['visa_validity_days'] = (flight_data['visaExpiryDate'] - flight_data['arrival_date']).dt.days
    flight_data['days_since_booking'] = (flight_data['arrival_date'] - flight_data['bookingDate']).dt.days
    flight_data['multiple_luggage'] = flight_data['travelData_noOfCheckingLuggage'] > 1
    flight_data['payment_to_arrival_gap'] = (flight_data['arrival_date'] - flight_data['paidParty_paidDate']).dt.days
    flight_data['recently_issued_passport'] = (flight_data['arrival_date'] - flight_data['travelDocument_expiryDate']).dt.days < 365
    high_risk_nationalities = ['TWN', 'PAK', 'FJI']
    flight_data['high_risk_nationalities'] = flight_data['travelDocument_issueCountry'].isin(high_risk_nationalities)

    max_arrival_date = flight_data['arrival_date'].max()
    timeframes = {
        'last_1_month': max_arrival_date - pd.DateOffset(months=1),
        'last_3_months': max_arrival_date - pd.DateOffset(months=3),
        'last_6_months': max_arrival_date - pd.DateOffset(months=6),
    }

    travel_counts = []
    for period_name, period_start in timeframes.items():
        count = (flight_data[flight_data['arrival_date'] >= period_start]
                 .groupby(['travelDocument_docNo', 'nationality'])['arrival_date']
                 .count().reset_index())
        count.columns = ['travelDocument_docNo', 'nationality', period_name]
        travel_counts.append(count)

    travel_counts_df = travel_counts[0]
    for count in travel_counts[1:]:
        travel_counts_df = travel_counts_df.merge(count, on=['travelDocument_docNo', 'nationality'], how='outer')

    flight_data = flight_data.merge(travel_counts_df, on=['travelDocument_docNo', 'nationality'], how='left')
    flight_data['last_trip_gap_days'] = flight_data.groupby(['travelDocument_docNo', 'nationality'])['arrival_date'].diff().dt.days
    flight_data['average_gap_between_trips_days'] = flight_data.groupby(['travelDocument_docNo', 'nationality'])['last_trip_gap_days'].transform('mean')
    flight_data['residence_vs_nationality_mismatch'] = flight_data['nationality'] != flight_data['residenceCountry']
    flight_data['multiple_concurrent_visas'] = flight_data.groupby(['travelDocument_docNo', 'nationality'])['visaNo'].transform('nunique') > 1

    # Replace all NaN values with zero
    flight_data.fillna(0, inplace=True)
    
    return flight_data

def encode_flight_data(featured_flight_data):
    
    # Ensure 'paxCount' is numeric, coercing errors to NaN
    featured_flight_data['paxCount'] = pd.to_numeric(featured_flight_data['paxCount'], errors='coerce')
    
    # Fill NaN values in 'paxCount' with 0
    featured_flight_data['paxCount'].fillna(0, inplace=True)
    
    # Check if 'gender' column exists before encoding
    if 'gender' in featured_flight_data.columns:
        gender_dummies = pd.get_dummies(featured_flight_data['gender'], prefix='gender', drop_first=False)
        featured_flight_data = pd.concat([featured_flight_data, gender_dummies], axis=1)
        featured_flight_data.drop(columns=['gender'], inplace=True)

    # Define boolean columns and check their existence
    boolean_columns = [
        'multiple_luggage', 'recently_issued_passport', 'high_risk_nationalities', 
        'residence_vs_nationality_mismatch', 'multiple_concurrent_visas', 
    ]
    existing_boolean_columns = [col for col in boolean_columns if col in featured_flight_data.columns]
    featured_flight_data[existing_boolean_columns] = featured_flight_data[existing_boolean_columns].astype(int)

    # Select final columns for the DataFrame
    selected_columns = [
        'passenger_id', 'paxCount', 'age', 
        'travel_document_validity_days', 'visa_validity_days', 'days_since_booking', 
        'payment_to_arrival_gap', 'last_1_month', 'last_3_months', 'last_6_months', 
        'last_trip_gap_days', 'average_gap_between_trips_days',
        'multiple_luggage', 'recently_issued_passport', 'high_risk_nationalities', 
        'residence_vs_nationality_mismatch', 'multiple_concurrent_visas',
        'gender_FEMALE', 'gender_MALE'
    ]
    # Filter existing columns
    selected_columns = [col for col in selected_columns if col in featured_flight_data.columns]
    
    featured_flight_data = featured_flight_data[selected_columns]
    
    # Set 'passenger_id' as the index
    featured_flight_data.set_index('passenger_id', inplace=True)

    return featured_flight_data

def clustering_and_anomaly_detection(encoded_flight_data, n_clusters=4, random_state=42, contamination=0.05, epochs=50, batch_size=32):
    
    # Define features for anomaly detection
    features = [
        'age', 'paxCount', 'multiple_luggage', 'travel_document_validity_days', 'visa_validity_days',
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 'last_3_months', 'last_6_months',
        'last_trip_gap_days', 'average_gap_between_trips_days', 'gender_FEMALE', 'gender_MALE',
        'recently_issued_passport', 'high_risk_nationalities', 'residence_vs_nationality_mismatch', 
        'multiple_concurrent_visas'
    ]
    
    # Select anomaly data based on defined features
    anomaly_data = encoded_flight_data[features].copy()
    
    # Define numerical columns for scaling
    numerical_columns = [
        'age', 'travel_document_validity_days', 'visa_validity_days', 
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 
        'last_3_months', 'last_6_months', 'last_trip_gap_days', 
        'average_gap_between_trips_days'
    ]
    
    # Define clustering features
    clustering_features = [
        'age', 'paxCount', 'travel_document_validity_days', 'visa_validity_days',
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 
        'last_3_months', 'last_6_months', 'last_trip_gap_days', 'multiple_luggage',
        'average_gap_between_trips_days', 'high_risk_nationalities', 
        'residence_vs_nationality_mismatch', 'multiple_concurrent_visas', 
        'recently_issued_passport', 'gender_FEMALE', 'gender_MALE'
    ]
    
    # Initialize the scaler
    scaler = StandardScaler()
    
    # Fit and transform the numerical data
    anomaly_data[numerical_columns] = scaler.fit_transform(anomaly_data[numerical_columns])
    
    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    anomaly_data['flight_cluster'] = kmeans.fit_predict(anomaly_data[clustering_features])

    # Define the features to be used for anomaly detection
    anomaly_features = [
        'age', 'paxCount', 'multiple_luggage', 'travel_document_validity_days', 'visa_validity_days',
        'days_since_booking', 'payment_to_arrival_gap', 'last_1_month', 'last_3_months', 'last_6_months',
        'last_trip_gap_days', 'average_gap_between_trips_days', 'gender_FEMALE', 'gender_MALE',
        'recently_issued_passport', 'high_risk_nationalities', 'residence_vs_nationality_mismatch', 
        'multiple_concurrent_visas', 'flight_cluster'
    ]

    # Select the relevant features for anomaly detection
    scaled_data = anomaly_data[anomaly_features]

    # Initialize the Isolation Forest model
    iso_forest = IsolationForest(contamination=contamination, random_state=42)  

    # Fit the model to the data
    iso_forest.fit(scaled_data)

    # Predict anomalies (1 = normal, -1 = anomaly)
    anomaly_predictions = iso_forest.predict(scaled_data)

    # Anomaly score for each point (higher score indicates more anomalous)
    anomaly_scores = iso_forest.decision_function(scaled_data)

    # Add anomaly predictions and scores to the dataset
    anomaly_data['anomaly_prediction'] = anomaly_predictions
    anomaly_data['anomaly_score'] = anomaly_scores

    # Define Autoencoder Model
    input_dim = scaled_data.shape[1]  # Number of features

    input_layer = Input(shape=(input_dim,))
    encoded = Dense(32, activation='relu')(input_layer)
    decoded = Dense(input_dim, activation='sigmoid')(encoded)

    autoencoder = Model(input_layer, decoded)

    # Compile and fit the model
    autoencoder.compile(optimizer=Adam(), loss='mse')
    autoencoder.fit(scaled_data, scaled_data, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

    # Get reconstruction error
    reconstructed = autoencoder.predict(scaled_data)
    reconstruction_error = np.mean(np.abs(scaled_data - reconstructed), axis=1)
    
    # Add reconstruction errors to the dataset
    anomaly_data['reconstruction_error'] = reconstruction_error

    # Define thresholds for each cluster based on reconstruction error (mean + 1.5 * std deviation)
    thresholds = anomaly_data.groupby('flight_cluster')['reconstruction_error'].agg(['mean', 'std'])
    thresholds['custom_threshold'] = thresholds['mean'] + 1.5 * thresholds['std']

    # Apply the threshold to each cluster
    def apply_threshold(row, thresholds):
        cluster = row['flight_cluster']
        return row['reconstruction_error'] > thresholds.loc[cluster, 'custom_threshold']

    anomaly_data['autoencoder_anomaly'] = anomaly_data.apply(apply_threshold, axis=1, thresholds=thresholds)

    # Combine the anomaly results into a final 'combined_anomaly' column
    anomaly_data['combined_anomaly'] = ((anomaly_data['anomaly_prediction'] == -1) | (anomaly_data['autoencoder_anomaly'] == 1)).astype(int)

    # Final anomaly score (use the maximum of the anomaly score and reconstruction error)
    anomaly_data['final_anomaly_score'] = np.maximum(anomaly_data['anomaly_score'], anomaly_data['reconstruction_error'])

    # Final anomalies output
    anomalies = anomaly_data[anomaly_data['combined_anomaly'] == 1][anomaly_features]

    # Reset index to convert passenger_id from index to a column in clustered_anomaly_data
    clustered_anomaly_data = anomalies.reset_index()

    # Perform the merge on passenger_id
    merged_arm_data = encoded_flight_data.merge(
        clustered_anomaly_data[['passenger_id']],  # Keep only passenger_id column
        on='passenger_id',
        how='inner'
    )

    # Set passenger_id as the index in the merged dataframe
    merged_arm_data.set_index('passenger_id', inplace=True)

    # Define the thresholds for binarization
    thresholds = {
        'age': 18,
        'paxCount': 2,
        'travel_document_validity_days': 30,
        'visa_validity_days': 30,
        'days_since_booking': 7,
        'payment_to_arrival_gap': 7,
        'last_trip_gap_days': 30,
        'average_gap_between_trips_days': 30
    }

    # Binarize features based on thresholds
    merged_arm_data['is_senior'] = (merged_arm_data['age'] >= thresholds['age']).astype(int)
    merged_arm_data['is_group_travel'] = (merged_arm_data['paxCount'] >= thresholds['paxCount']).astype(int)
    merged_arm_data['is_travel_document_expiring'] = (merged_arm_data['travel_document_validity_days'] < thresholds['travel_document_validity_days']).astype(int)
    merged_arm_data['is_visa_expiring'] = (merged_arm_data['visa_validity_days'] < thresholds['visa_validity_days']).astype(int)
    merged_arm_data['is_last_minute_booking'] = (merged_arm_data['days_since_booking'] < thresholds['days_since_booking']).astype(int)
    merged_arm_data['is_last_minute_payment'] = (merged_arm_data['payment_to_arrival_gap'] < thresholds['payment_to_arrival_gap']).astype(int)
    merged_arm_data['is_frequent_travel'] = (merged_arm_data['last_trip_gap_days'] < thresholds['average_gap_between_trips_days']).astype(int)
    merged_arm_data['is_high_avg_freq_travel'] = (merged_arm_data['average_gap_between_trips_days'] < thresholds['average_gap_between_trips_days']).astype(int)

    # Define thresholds for last_n_months features
    months_thresholds = {
        'last_1_month': 3,  # 3 trips or more in the last month could be a red flag
        'last_3_months': 5,  # 5 trips or more in the last 3 months could indicate a pattern shift
        'last_6_months': 7   # 7 trips or more in the last 6 months might indicate frequent travel
    }

    # Binarize last n months features
    merged_arm_data['is_frequent_travel_last_1_month'] = (merged_arm_data['last_1_month'] >= months_thresholds['last_1_month']).astype(int)
    merged_arm_data['is_frequent_travel_last_3_month'] = (merged_arm_data['last_3_months'] >= months_thresholds['last_3_months']).astype(int)
    merged_arm_data['is_frequent_travel_last_6_month'] = (merged_arm_data['last_6_months'] >= months_thresholds['last_6_months']).astype(int)

    # Drop original columns after binarization
    merged_arm_data = merged_arm_data.drop(columns=['age', 'paxCount', 'travel_document_validity_days',
                                                   'visa_validity_days', 'days_since_booking', 'payment_to_arrival_gap',
                                                   'last_1_month', 'last_3_months', 'last_6_months', 'last_trip_gap_days',
                                                   'average_gap_between_trips_days'])

    # Convert all columns to boolean (True/False) for optimal performance in mlxtend
    merged_arm_data = merged_arm_data.astype(bool)

    return merged_arm_data

def association_rule_mining(encoded_flight_data, anomaly_data):
    
    # Apply the FP-Growth algorithm
    frequent_itemsets = fpgrowth(anomaly_data, min_support=0.01, use_colnames=True)

    # Generate association rules using lift as the metric
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

    # Sort rules by confidence and lift
    sorted_rules = rules.sort_values(by=['confidence', 'lift'], ascending=False)

    # Initialize variables to track distinct rules and used features
    used_features = set()
    distinct_rules = []

    # Define travel count features to ensure they do not appear together
    travel_count_features = {'is_frequent_travel_last_1_month', 'is_frequent_travel_last_3_month', 'is_frequent_travel_last_6_month',
                             'is_frequent_travel', 'is_high_avg_freq_travel'}

    # Iterate through sorted rules to find three distinct rules
    for _, rule in sorted_rules.iterrows():
        antecedents = set(rule['antecedents'])
        consequents = set(rule['consequents'])

        # Combine antecedents and consequents to check for travel count features
        all_features = antecedents | consequents

        # Check if any of the travel count features appear together
        if len(all_features & travel_count_features) > 1:
            continue  # Skip if more than one travel count feature is present

        # Ensure no overlap with used features
        if not (antecedents & used_features) and not (consequents & used_features):
            distinct_rules.append(rule)
            used_features.update(all_features)

        if len(distinct_rules) == 3:
            break

    # Convert the distinct rules into a DataFrame
    distinct_rules_df = pd.DataFrame(distinct_rules)
    
    return distinct_rules_df

def flagged_passenger_details(rules_df, anomaly_data):

    # Feature map for human-readable feature names
    feature_map = {
        'multiple_luggage': 'carry multiple pieces of luggage',
        'recently_issued_passport': 'have a recently issued passport',
        'high_risk_nationalities': 'belong to a high-risk nationality',
        'residence_vs_nationality_mismatch': 'have a mismatch between residence country and nationality',
        'multiple_concurrent_visas': 'hold multiple concurrent visas',
        'gender_FEMALE': 'are female',
        'gender_MALE': 'are male',
        'is_senior': 'are an adult (above 18)',
        'is_group_travel': 'are part of a group travel',
        'is_travel_document_expiring': 'have a travel document with less than 30 days of validity',
        'is_visa_expiring': 'have a visa with less than 30 days of validity',
        'is_last_minute_booking': 'booked travel less than 7 days before the trip',
        'is_last_minute_payment': 'made payment less than 7 days before the trip',
        'is_frequent_travel': 'with a last trip gap of less than 30 days',
        'is_high_avg_freq_travel': 'with an average historical travel gap of less than 30 days',
        'is_frequent_travel_last_1_month': 'traveled 3 or more times in the last month',
        'is_frequent_travel_last_3_month': 'traveled 5 or more times in the last 3 months',
        'is_frequent_travel_last_6_month': 'traveled 7 or more times in the last 6 months'
    }
    # Generate readable rule sentences
    sentences = []
    
    # Initialize the counter for rule numbering
    rule_number = 1
    
    for _, row in rules_df.iterrows():
        # Extract antecedents and consequents as frozensets
        antecedents = row['antecedents']
        consequents = row['consequents']
        
        # Ensure antecedents and consequents are in a list form (converting frozenset if needed)
        if isinstance(antecedents, frozenset):
            antecedents = list(antecedents)
        if isinstance(consequents, frozenset):
            consequents = list(consequents)
        
        # Convert antecedents and consequents to readable conditions
        antecedent_conditions = [feature_map.get(str(antecedent), antecedent) for antecedent in antecedents]
        consequent_conditions = [feature_map.get(str(consequent), consequent) for consequent in consequents]
        
        # Create the rule sentence with custom numbering
        sentence = f"Rule {rule_number}: Passengers who " + ", ".join(antecedent_conditions + consequent_conditions) + " are identified as high-risk passengers."
        sentences.append(sentence)

        # Increment the rule number for the next rule
        rule_number += 1

    # Ensure 'antecedents' and 'consequents' are in a proper format for filtering
    rules_df['antecedents'] = rules_df['antecedents'].apply(lambda x: list(x))
    rules_df['consequents'] = rules_df['consequents'].apply(lambda x: list(x))

    # Initialize a list to store flagged passengers
    flagged_passengers = []

    # Initialize a rule counter (assuming the numbering starts from 1)
    rule_number = 1

    # Iterate over each rule to filter passengers
    for _, rule in rules_df.iterrows():
        # Combine antecedents and consequents for filtering
        conditions = rule['antecedents'] + rule['consequents']
        
        # Filter passengers who meet all conditions in the current rule
        mask = anomaly_data[conditions].all(axis=1)
        flagged_passenger_ids = anomaly_data[mask].index.tolist()
        
        # Append passenger IDs and their corresponding rule number to the list
        for pid in flagged_passenger_ids:
            flagged_passengers.append({
                'passenger_id': pid,
                'rule_number': rule_number  # Append the rule number
            })

        # Increment the rule number for the next rule
        rule_number += 1

    # Convert the list of flagged passengers into a DataFrame
    flagged_passengers_df = pd.DataFrame(flagged_passengers)

    # Group flagged passengers by passenger_id and aggregate rule numbers into a comma-separated string
    flagged_passengers_grouped = flagged_passengers_df.groupby('passenger_id')['rule_number'].apply(lambda x: ', '.join(map(str, x))).reset_index()
    
    # Rename the aggregated column for clarity
    flagged_passengers_grouped.rename(columns={'rule_number': 'Flagged_Rule_Numbers'}, inplace=True)
    
    # Select the required columns from the 'featured_flight_data' dataframe
    selected_columns = ['givenName', 'lastName', 'dob', 'gender', 'nationality', 'travelDocument_docNo', 'visaNo', 'residenceCountry', 'passenger_id']
    data_selected = flight_data[selected_columns]
    
    # Merge 'flagged_passengers_grouped' with 'data_selected' on passenger_id
    final_merged_df = data_selected.merge(flagged_passengers_grouped, on='passenger_id', how='inner')
    
    # Rename the columns (modify the new names as per your requirement)
    final_merged_df = final_merged_df.rename(columns={
    'givenName': 'Given_Name',
    'lastName': 'Last_Name',
    'dob': 'Date_of_Birth',
    'gender': 'Gender',
    'nationality': 'Nationality',
    'travelDocument_docNo': 'Travel_Document_Number',
    'visaNo': 'Visa_Number',
    'residenceCountry': 'Residence_Country'})

    return sentences,final_merged_df



# Custom Environment
class CustomEnv(gym.Env):
    def __init__(self, rules_df):
        super(CustomEnv, self).__init__()
        
        # Define action space and observation space
        self.action_space = spaces.Discrete(3)  # Example: 3 actions
        self.observation_space = spaces.Discrete(10)  # Example: 10 different observations (modify as per your case)
        
        # Store rules
        self.rules_df = rules_df  # This is where your association rules dataframe is passed
        self.current_step = 0

    def reset(self):
        self.current_step = 0
        # Return an initial state
        return self.current_step  # Example: initial state is just the current_step
    
    def step(self, action):
        # Example step logic based on your rules and actions
        # Action interpretation depends on your custom logic
        reward = 0
        
        # Perform action and calculate reward
        if action == 0:
            reward = self.apply_rule_1()  # Dummy method to simulate applying a rule
        elif action == 1:
            reward = self.apply_rule_2()  # Dummy method to simulate applying a rule
        else:
            reward = self.apply_rule_3()  # Dummy method to simulate applying a rule

        # Proceed to next state
        self.current_step += 1
        done = self.current_step >= 10  # Example: end after 10 steps
        
        return self.current_step, reward, done, {}

    def apply_rule_1(self):
        # Example of how you would use the association rules for rewards
        # Look up some rules, and compute reward (just a dummy implementation)
        return 1
    
    def apply_rule_2(self):
        # Another rule application logic
        return 2
    
    def apply_rule_3(self):
        # Another rule application logic
        return -1


The above method uses a **semi-static rule creation approach**, with some dynamic elements based on data patterns. Here's a breakdown:

**Static Elements:**

- Thresholds for Binarization:
The thresholds for age, travel document validity, visa validity, etc., are pre-defined and hardcoded in the thresholds and months_thresholds dictionaries. These thresholds are static and need to be manually updated if travel patterns or risks evolve.

- Feature Mapping:
The feature_map that translates feature names into human-readable descriptions is fixed and does not change based on data patterns or new features.

- Predefined Feature Combinations:
The conditions for generating association rules assume fixed features like "is_group_travel" or "is_senior" and predefined binarized columns.

**Dynamic Elements:**

- FP-Growth Algorithm:
The FP-Growth algorithm dynamically identifies frequent itemsets from the encoded data based on patterns that emerge in the passenger data. This allows the model to discover associations between features (like frequent travel and expiring visas) without predefining those associations.

- Filtering of Rules:
Rules are sorted and filtered dynamically based on metrics like confidence and lift, ensuring only the strongest and most relevant rules are used. This adds a layer of adaptability.

- Passenger Filtering:
Flagging of passengers based on the generated rules happens dynamically, as the algorithm checks which passengers meet the conditions for each rule.

### Filter flight_id s 

The resulting DataFrame containin unique flight IDs along with their corresponding arrival month and year for further analysis.

In [4]:
# Filter flightDetails__id for the specified month with their corresponding flightDetails_arrivalTime
filtered_data = data[data['arrival_month_year'] == '2024-10']
distinct_flight_details = filtered_data[['flightDetails__id', 'arrival_month_year']].drop_duplicates()
distinct_flight_details

,flightDetails__id,arrival_month_year
57167,AI273-2024-10-09T04:30-MAA-CMB,2024-10
57493,UL504-2024-10-09T03:00-LHR-CMB,2024-10
57817,UL116-2024-10-09T15:20-MLE-CMB,2024-10
58138,EK652-2024-10-09T12:45-DXB-CMB,2024-10
59110,UL605-2024-10-09T07:10-MEL-CMB,2024-10
59438,UL186-2024-10-09T15:30-LHE-CMB,2024-10
59761,EK506-2024-10-22T09:15-DXB-KUL,2024-10
60082,MH277-2024-10-22T09:45-LHR-KUL,2024-10
60410,SG488-2024-10-22T10:00-SIN-KUL,2024-10
60736,TG106-2024-10-22T10:15-BKK-KUL,2024-10


We can include below def function calls in the backend API to process the specific flight data and apply the above pipeline logic. The process would look like this:

- **Flight Data Retrieval:** The backend would first retrieve the flight data based on a specific flight_ID, using a query to filter the relevant flight details.

- **Feature Engineering:** The _feature_engineering(flight_data)_ function would process the flight data to generate new features that are essential for the predictive model.

- **Data Encoding:** The _encode_flight_data(featured_flight_data)_ function would then encode the processed data for machine learning.

- **Clustering and Anomaly Detection:** The _clustering_and_anomaly_detection(encoded_flight_data)_ function would apply clustering (e.g., KMeans) and detect anomalies within the encoded data.

- **Association Rule Mining:** Finally, the _association_rule_mining(encoded_flight_data, clustered_anomaly_data, featured_flight_data)_ function would generate the association rules and merge the results with the original data.

**These functions can be executed as part of the API request handling process, with the final results (sentences, final_merged_df) being sent back to the client, such as a frontend or another system, in the form of a JSON response.**

<div style="text-align: left;">
    <h3> </font> <font color = #00BFFF>04- Example Usage</h3> </font>
</div>

#### 01 

- Flight Number:TK8573
- Arrive Date & Time: 2024-10-24
- Route: IST-TAS

In [5]:
# Example of the specific flight details
specific_flight_ID = 'TK8573-2024-10-24T06:00-IST-TAS'
flight_data = data[(data['flightDetails__id'] == specific_flight_ID)]

# Calling above def functions
featured_flight_data= feature_engineering(flight_data)
encoded_flight_data = encode_flight_data(featured_flight_data)
anomaly_data = clustering_and_anomaly_detection(encoded_flight_data)
rules_df = association_rule_mining(encoded_flight_data, anomaly_data)
sentences,final_merged_df= flagged_passenger_details(rules_df, anomaly_data)
sentences
final_merged_df

# Load the saved PPO model
ppo_model = PPO.load("ppo_custom_env_model")

# Initialize the custom environment
env = CustomEnv(rules_df)
env = DummyVecEnv([lambda: env])

# To evaluate the model
obs = env.reset()
# Run the model for 10 steps (or as long as desired)
for _ in range(10):
    action, _state = ppo_model.predict(obs)  # Use the model to predict the next action
    obs, reward, done, info = env.step(action)  # Take a step in the environment

    if done:
        print("Episode finished.")
        break
    print(f"Step: {_}, Action: {action}, Reward: {reward}")

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 0.6527 - val_loss: 0.5426
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6321 - val_loss: 0.5118
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.5571 - val_loss: 0.4833
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.5097 - val_loss: 0.4570
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.5735 - val_loss: 0.4322
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.5022 - val_loss: 0.4093
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.4444 - val_loss: 0.3882
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.4519 - val_loss: 0.3698
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.4660 - val_loss: 0.3539
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.3734 - val_loss: 0.3407
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.3815 - val_loss: 0.3300
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.3

['Rule 1: Passengers who are part of a group travel, are female, have a visa with less than 30 days of validity are identified as high-risk passengers.',
 'Rule 2: Passengers who are an adult (above 18), have a travel document with less than 30 days of validity, belong to a high-risk nationality, carry multiple pieces of luggage are identified as high-risk passengers.',
 'Rule 3: Passengers who have a recently issued passport, with a last trip gap of less than 30 days are identified as high-risk passengers.']

,Given_Name,Last_Name,Date_of_Birth,Gender,Nationality,Travel_Document_Number,Visa_Number,Residence_Country,passenger_id,Flagged_Rule_Numbers
0,FAISAL,BIN ABD WAHAB,2018-10-17,MALE,GBR,S2121183,VP2023196,GBR,221,3
1,ROBERT,BIN ABAS,1976-08-10,MALE,FJI,U1266074,VP2023961,FJI,198,3
2,LILY,AHMAD,2013-06-08,FEMALE,USA,J1215251,VP2023945,USA,229,3
3,GEORG,OOI,2005-11-27,MALE,USA,D2014234,VP2023677,USA,302,3
4,HANNA,ARIFF,2013-06-12,FEMALE,FJI,S1940314,VP2023647,FJI,321,3
5,HARUN,BINTI MAMAT,1975-04-04,FEMALE,PAK,F1926624,VP2023360,PAK,310,3
6,IMRAN,BINTI DOLLAH,1996-12-04,FEMALE,PAK,E2195543,VP2023531,PAK,307,"2, 3"
7,SAM,ARIFF,2016-08-05,MALE,AUS,Q3847466,VP2023527,AUS,306,3
8,HANNA,DARUS,2018-05-15,FEMALE,FJI,V2270946,VP2023402,FJI,304,3
9,JENNIFER,ARIFF,2004-09-19,FEMALE,TWN,S2852987,VP2023451,TWN,144,"1, 3"


Step: 0, Action: [2], Reward: [-1.]
Step: 1, Action: [2], Reward: [-1.]
Step: 2, Action: [1], Reward: [2.]
Step: 3, Action: [0], Reward: [1.]
Step: 4, Action: [0], Reward: [1.]
Step: 5, Action: [1], Reward: [2.]
Step: 6, Action: [1], Reward: [2.]
Step: 7, Action: [1], Reward: [2.]
Step: 8, Action: [2], Reward: [-1.]
Episode finished.


#### 02

- Flight Number:EK652
- Arrive Date & Time: 2024-10-22
- Route: DXB-KUL

In [6]:
# Example of the specific flight details
specific_flight_ID = 'QF523-2024-10-22T08:45-DXB-KUL'
flight_data = data[(data['flightDetails__id'] == specific_flight_ID)]

# Calling above def functions
featured_flight_data= feature_engineering(flight_data)
encoded_flight_data = encode_flight_data(featured_flight_data)
anomaly_data = clustering_and_anomaly_detection(encoded_flight_data)
rules_df = association_rule_mining(encoded_flight_data, anomaly_data)
sentences,final_merged_df= flagged_passenger_details(rules_df, anomaly_data)
sentences
final_merged_df

# Load the saved PPO model
ppo_model = PPO.load("ppo_custom_env_model")

# Initialize the custom environment
env = CustomEnv(rules_df)
env = DummyVecEnv([lambda: env])

# To evaluate the model
obs = env.reset()
# Run the model for 10 steps (or as long as desired)
for _ in range(10):
    action, _state = ppo_model.predict(obs)  # Use the model to predict the next action
    obs, reward, done, info = env.step(action)  # Take a step in the environment

    if done:
        print("Episode finished.")
        break
    print(f"Step: {_}, Action: {action}, Reward: {reward}")

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 0.6661 - val_loss: 0.4909
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6128 - val_loss: 0.4640
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.5753 - val_loss: 0.4380
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.5507 - val_loss: 0.4122
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.5473 - val_loss: 0.3874
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.4893 - val_loss: 0.3635
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4209 - val_loss: 0.3412
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.4716 - val_loss: 0.3216
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.4197 - val_loss: 0.3049
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4520 - val_loss: 0.2909
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.3625 - val_loss: 0.2793
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.3

['Rule 1: Passengers who are part of a group travel, belong to a high-risk nationality, are female, have a visa with less than 30 days of validity are identified as high-risk passengers.',
 'Rule 2: Passengers who have a recently issued passport, with a last trip gap of less than 30 days are identified as high-risk passengers.',
 'Rule 3: Passengers who are an adult (above 18), with an average historical travel gap of less than 30 days are identified as high-risk passengers.']

,Given_Name,Last_Name,Date_of_Birth,Gender,Nationality,Travel_Document_Number,Visa_Number,Residence_Country,passenger_id,Flagged_Rule_Numbers
0,SAM,BIN ABD WAHAB,2001-06-12,FEMALE,LKA,M3176440,VP2023024,LKA,269,"2, 3"
1,DIYA,LEO,2019-07-17,MALE,IND,G3885147,VP2023905,IND,111,2
2,DIYA,DARUS,1996-05-14,FEMALE,FJI,L2518850,VP2023611,FJI,106,"2, 3"
3,ROBERT,OOI,2020-01-22,FEMALE,USA,O3513857,VP2023881,USA,53,2
4,SAM,AHMAD,1989-08-03,MALE,PAK,R1351869,VP2023548,PAK,56,"2, 3"
5,SAM,GOH,1978-12-03,MALE,PAK,O1318865,VP2023701,PAK,276,"2, 3"
6,JENNIFER,ARIFF,2004-09-19,FEMALE,TWN,S2852987,VP2023451,TWN,144,"1, 2, 3"
7,ANN,MOHAMED,2020-10-26,FEMALE,USA,Z3111188,VP2023417,USA,153,2
8,JOHN,ARIFF,1968-10-06,MALE,AUS,D1113196,VP2023276,AUS,140,"2, 3"
9,MIKE,KADIR,2014-02-09,MALE,FJI,J3106881,VP2023889,FJI,192,2


Step: 0, Action: [0], Reward: [1.]
Step: 1, Action: [1], Reward: [2.]
Step: 2, Action: [1], Reward: [2.]
Step: 3, Action: [1], Reward: [2.]
Step: 4, Action: [1], Reward: [2.]
Step: 5, Action: [1], Reward: [2.]
Step: 6, Action: [0], Reward: [1.]
Step: 7, Action: [0], Reward: [1.]
Step: 8, Action: [2], Reward: [-1.]
Episode finished.


#### 03

- Flight Number:EK652
- Arrive Date & Time: 2024-10-09
- Route: DXB-CMB

In [7]:
# Example of the specific flight details
specific_flight_ID = 'EK652-2024-10-09T12:45-DXB-CMB'
flight_data = data[(data['flightDetails__id'] == specific_flight_ID)]

# Calling above def functions
featured_flight_data= feature_engineering(flight_data)
encoded_flight_data = encode_flight_data(featured_flight_data)
anomaly_data = clustering_and_anomaly_detection(encoded_flight_data)
rules_df = association_rule_mining(encoded_flight_data, anomaly_data)
sentences,final_merged_df= flagged_passenger_details(rules_df, anomaly_data)
sentences
final_merged_df

# Load the saved PPO model
ppo_model = PPO.load("ppo_custom_env_model")

# Initialize the custom environment
env = CustomEnv(rules_df)
env = DummyVecEnv([lambda: env])

# To evaluate the model
obs = env.reset()
# Run the model for 10 steps (or as long as desired)
for _ in range(10):
    action, _state = ppo_model.predict(obs)  # Use the model to predict the next action
    obs, reward, done, info = env.step(action)  # Take a step in the environment

    if done:
        print("Episode finished.")
        break
    print(f"Step: {_}, Action: {action}, Reward: {reward}")

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - loss: 0.6204 - val_loss: 0.4933
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.6111 - val_loss: 0.4665
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.6065 - val_loss: 0.4414
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.5477 - val_loss: 0.4174
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.5001 - val_loss: 0.3938
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4725 - val_loss: 0.3699
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.4328 - val_loss: 0.3472
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.4545 - val_loss: 0.3260
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4199 - val_loss: 0.3073
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.4309 - val_loss: 0.2916
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.4035 - val_loss: 0.2789
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.3

['Rule 1: Passengers who booked travel less than 7 days before the trip, are part of a group travel, are female are identified as high-risk passengers.',
 'Rule 2: Passengers who have a visa with less than 30 days of validity, are an adult (above 18), carry multiple pieces of luggage, are male are identified as high-risk passengers.',
 'Rule 3: Passengers who have a recently issued passport, with a last trip gap of less than 30 days are identified as high-risk passengers.']

,Given_Name,Last_Name,Date_of_Birth,Gender,Nationality,Travel_Document_Number,Visa_Number,Residence_Country,passenger_id,Flagged_Rule_Numbers
0,VIGNARAJAH,MANOHARAN,1975-09-03,MALE,LKA,N5330233,VP2023048,LKA,50,3
1,DIYA,DARUS,1996-05-14,FEMALE,FJI,L2518850,VP2023611,FJI,106,3
2,SAM,AHMAD,1989-08-03,MALE,PAK,R1351869,VP2023548,PAK,56,3
3,SAM,GOH,1978-12-03,MALE,PAK,O1318865,VP2023701,PAK,276,3
4,JENNIFER,ARIFF,2004-09-19,FEMALE,TWN,S2852987,VP2023451,TWN,144,3
...,...,...,...,...,...,...,...,...,...,...
67,JOHN,ARIFF,1968-10-06,MALE,AUS,D1113196,VP2023276,AUS,140,3
68,THIBIKA,JEGANATHAN,2007-11-20,FEMALE,LKA,N8475269,VP2023542,LKA,141,"1, 3"
69,SAM,GOH,1978-12-03,MALE,PAK,O1318865,VP2023701,PAK,276,3
70,SAM,AHMAD,1989-08-03,MALE,PAK,R1351869,VP2023548,PAK,56,3


Step: 0, Action: [0], Reward: [1.]
Step: 1, Action: [1], Reward: [2.]
Step: 2, Action: [1], Reward: [2.]
Step: 3, Action: [1], Reward: [2.]
Step: 4, Action: [1], Reward: [2.]
Step: 5, Action: [2], Reward: [-1.]
Step: 6, Action: [1], Reward: [2.]
Step: 7, Action: [1], Reward: [2.]
Step: 8, Action: [1], Reward: [2.]
Episode finished.


#### 04

- Flight Number:UL186
- Arrive Date & Time: 2024-10-09
- Route: LHE-CMB

In [8]:
# Example of the specific flight details
specific_flight_ID = 'UL186-2024-10-09T15:30-LHE-CMB'
flight_data = data[(data['flightDetails__id'] == specific_flight_ID)]

# Calling above def functions
featured_flight_data= feature_engineering(flight_data)
encoded_flight_data = encode_flight_data(featured_flight_data)
anomaly_data = clustering_and_anomaly_detection(encoded_flight_data)
rules_df = association_rule_mining(encoded_flight_data, anomaly_data)
sentences,final_merged_df= flagged_passenger_details(rules_df, anomaly_data)
sentences
final_merged_df

# Load the saved PPO model
ppo_model = PPO.load("ppo_custom_env_model")

# Initialize the custom environment
env = CustomEnv(rules_df)
env = DummyVecEnv([lambda: env])

# To evaluate the model
obs = env.reset()
# Run the model for 10 steps (or as long as desired)
for _ in range(10):
    action, _state = ppo_model.predict(obs)  # Use the model to predict the next action
    obs, reward, done, info = env.step(action)  # Take a step in the environment

    if done:
        print("Episode finished.")
        break
    print(f"Step: {_}, Action: {action}, Reward: {reward}")

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.5846 - val_loss: 0.5096
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.5877 - val_loss: 0.4768
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.5559 - val_loss: 0.4473
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.5625 - val_loss: 0.4200
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4918 - val_loss: 0.3958
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.4998 - val_loss: 0.3738
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.4386 - val_loss: 0.3544
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.4310 - val_loss: 0.3380
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.4802 - val_loss: 0.3242
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4173 - val_loss: 0.3122
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.3996 - val_loss: 0.3020
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3

['Rule 1: Passengers who are part of a group travel, belong to a high-risk nationality, are female, have a visa with less than 30 days of validity are identified as high-risk passengers.',
 'Rule 2: Passengers who have a recently issued passport, with a last trip gap of less than 30 days are identified as high-risk passengers.',
 'Rule 3: Passengers who are an adult (above 18), with an average historical travel gap of less than 30 days are identified as high-risk passengers.']

,Given_Name,Last_Name,Date_of_Birth,Gender,Nationality,Travel_Document_Number,Visa_Number,Residence_Country,passenger_id,Flagged_Rule_Numbers
0,DIYA,DARUS,1996-05-14,FEMALE,FJI,L2518850,VP2023611,FJI,106,"2, 3"
1,DIYA,LEO,2019-07-17,MALE,IND,G3885147,VP2023905,IND,111,2
2,ROBERT,OOI,2020-01-22,FEMALE,USA,O3513857,VP2023881,USA,53,2
3,SAM,AHMAD,1989-08-03,MALE,PAK,R1351869,VP2023548,PAK,56,"2, 3"
4,DIYA,BINTI DOLLAH,2009-08-21,MALE,USA,Y3614662,VP2023829,USA,62,2
5,SAM,GOH,1978-12-03,MALE,PAK,O1318865,VP2023701,PAK,276,"2, 3"
6,ANN,KADIR,2004-11-13,FEMALE,LKA,C2481178,VP2023661,LKA,274,"2, 3"
7,ALI,GOH,2010-04-22,MALE,IND,B1784421,VP2023200,IND,126,2
8,VIGNARAJAH,MANOHARAN,1975-09-03,MALE,LKA,N5330233,VP2023048,LKA,50,"2, 3"
9,JENNIFER,ARIFF,2004-09-19,FEMALE,TWN,S2852987,VP2023451,TWN,144,"1, 2, 3"


Step: 0, Action: [1], Reward: [2.]
Step: 1, Action: [1], Reward: [2.]
Step: 2, Action: [0], Reward: [1.]
Step: 3, Action: [0], Reward: [1.]
Step: 4, Action: [0], Reward: [1.]
Step: 5, Action: [1], Reward: [2.]
Step: 6, Action: [1], Reward: [2.]
Step: 7, Action: [1], Reward: [2.]
Step: 8, Action: [1], Reward: [2.]
Episode finished.


#### 04

- Flight Number:TG106
- Arrive Date & Time: 2024-10-22
- Route: BKK-KUL

In [9]:
# Example of the specific flight details
specific_flight_ID = 'TG106-2024-10-22T10:15-BKK-KUL'
flight_data = data[(data['flightDetails__id'] == specific_flight_ID)]

# Calling above def functions
featured_flight_data= feature_engineering(flight_data)
encoded_flight_data = encode_flight_data(featured_flight_data)
anomaly_data = clustering_and_anomaly_detection(encoded_flight_data)
rules_df = association_rule_mining(encoded_flight_data, anomaly_data)
sentences,final_merged_df= flagged_passenger_details(rules_df, anomaly_data)
sentences
final_merged_df

# Load the saved PPO model
ppo_model = PPO.load("ppo_custom_env_model")

# Initialize the custom environment
env = CustomEnv(rules_df)
env = DummyVecEnv([lambda: env])

# To evaluate the model
obs = env.reset()
# Run the model for 10 steps (or as long as desired)
for _ in range(10):
    action, _state = ppo_model.predict(obs)  # Use the model to predict the next action
    obs, reward, done, info = env.step(action)  # Take a step in the environment

    if done:
        print("Episode finished.")
        break
    print(f"Step: {_}, Action: {action}, Reward: {reward}")

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.7054 - val_loss: 0.5904
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6588 - val_loss: 0.5610
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6082 - val_loss: 0.5335
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.5969 - val_loss: 0.5076
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.5467 - val_loss: 0.4823
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.4966 - val_loss: 0.4587
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.5666 - val_loss: 0.4359
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.4865 - val_loss: 0.4142
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.4536 - val_loss: 0.3943
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4071 - val_loss: 0.3764
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.4189 - val_loss: 0.3603
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3

['Rule 1: Passengers who are part of a group travel, are female, have a visa with less than 30 days of validity are identified as high-risk passengers.',
 'Rule 2: Passengers who belong to a high-risk nationality, have a travel document with less than 30 days of validity, are an adult (above 18), carry multiple pieces of luggage are identified as high-risk passengers.',
 'Rule 3: Passengers who have a recently issued passport, with a last trip gap of less than 30 days are identified as high-risk passengers.']

,Given_Name,Last_Name,Date_of_Birth,Gender,Nationality,Travel_Document_Number,Visa_Number,Residence_Country,passenger_id,Flagged_Rule_Numbers
0,ROBERT,OOI,2016-09-17,MALE,FJI,H2510157,VP2023864,FJI,224,3
1,LILY,AHMAD,2013-06-08,FEMALE,USA,J1215251,VP2023945,USA,229,3
2,GABRIEL,BINTI DOLLAH,2001-12-14,MALE,FJI,Q2238673,VP2023287,FJI,334,3
3,ERIC,KADIR,2018-04-10,FEMALE,PAK,K2143340,VP2023222,PAK,330,3
4,SAM,BIN ABD WAHAB,2001-06-12,FEMALE,LKA,M3176440,VP2023024,LKA,269,3
5,HARUN,BINTI MAMAT,1975-04-04,FEMALE,PAK,F1926624,VP2023360,PAK,310,3
6,IMRAN,BINTI DOLLAH,1996-12-04,FEMALE,PAK,E2195543,VP2023531,PAK,307,"2, 3"
7,SAM,ARIFF,2016-08-05,MALE,AUS,Q3847466,VP2023527,AUS,306,3
8,HANNA,DARUS,2018-05-15,FEMALE,FJI,V2270946,VP2023402,FJI,304,3
9,SAM,LEO,2018-12-21,FEMALE,FJI,Y3863835,VP2023270,FJI,318,3


Step: 0, Action: [0], Reward: [1.]
Step: 1, Action: [0], Reward: [1.]
Step: 2, Action: [0], Reward: [1.]
Step: 3, Action: [1], Reward: [2.]
Step: 4, Action: [0], Reward: [1.]
Step: 5, Action: [1], Reward: [2.]
Step: 6, Action: [1], Reward: [2.]
Step: 7, Action: [0], Reward: [1.]
Step: 8, Action: [1], Reward: [2.]
Episode finished.
